Loading the Libraries


In [1]:
import h5py
import pandas as pd
import numpy as np
import os

file paths

In [3]:

# labels = pd.read_csv("/Users/vaidehipandya/Desktop/Work/HPC_labels_subset.csv")
# latent = pd.read_csv("/Users/vaidehipandya/Desktop/Work/exports/train_img_h_latent.csv")


# ==== CONFIG: INPUT FILES FOR KB CLEANING ====

BASE_DIR = "/Users/vaidehipandya/Desktop/Work"     

# Main representation file
H5_FILE = f"{BASE_DIR}/csv_files/from_raw_data/subset_all_10000.h5"

# Annotation files (three separate CSVs)
ANNOTATION_FILES = [
    f"{BASE_DIR}/csv_files/from_raw_data/pa_1.csv",
    f"{BASE_DIR}/csv_files/from_raw_data/pa_2.csv",
    f"{BASE_DIR}/csv_files/from_raw_data/pa_3.csv"
]

# Label file
TILE_LABEL_FILE = f"{BASE_DIR}/csv_files/from_raw_data/HPC_lables.csv"
LABEL_SUBSET = f"{BASE_DIR}/csv_files/For_KB/HPC_labels_subset.csv"
#Survival file
SURVIVAL = f"{BASE_DIR}/csv_files/from_raw_data/all_folds_summary.csv"



In [4]:


# Load annotations
annotation_dfs = [pd.read_csv(f) for f in ANNOTATION_FILES]
annotations = pd.concat(annotation_dfs, ignore_index=True)

# Load label file
labels = pd.read_csv(TILE_LABEL_FILE)
labels_ss = pd.read_csv(LABEL_SUBSET)
# Load survival file
survival = pd.read_csv(SURVIVAL)

In [5]:



OUTPUT_DIR = "./h5_exports"  # where CSVs will be saved

os.makedirs(OUTPUT_DIR, exist_ok=True)

with h5py.File(H5_FILE, "r") as f:
    
    print("Datasets found:", list(f.keys()))
    
    for key in f.keys():
        data = f[key][:]
        
        print(f"Processing dataset: {key} with shape {data.shape}")
        
        # If dataset is 1D → simple CSV
        if data.ndim == 1:
            df = pd.DataFrame({key: data})
        
        # If dataset is 2D → treat rows as embeddings / arrays
        elif data.ndim == 2:
            df = pd.DataFrame(data)
        
        else:
            # If it is weird shape, flatten safely
            df = pd.DataFrame(data.reshape(data.shape[0], -1))

        # Convert bytes → strings if needed
        df = df.applymap(lambda x: x.decode("utf-8") if isinstance(x, bytes) else x)

        # Save CSV
        out_path = os.path.join(OUTPUT_DIR, f"{key}.csv")
        df.to_csv(out_path, index=False)
        
        print(f"Saved → {out_path}")

print("Done! All datasets extracted.")

Datasets found: ['train_hist_subtype', 'train_img_h_latent', 'train_img_z_latent', 'train_labels', 'train_patterns', 'train_samples', 'train_slides', 'train_tiles']
Processing dataset: train_hist_subtype with shape (10000,)
Saved → ./h5_exports/train_hist_subtype.csv
Processing dataset: train_img_h_latent with shape (10000, 1024)


/var/folders/q4/sypcwz3j32b0qyjm3q7rk6cc0000gn/T/ipykernel_52113/3322175096.py:27: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.decode("utf-8") if isinstance(x, bytes) else x)


Saved → ./h5_exports/train_img_h_latent.csv
Processing dataset: train_img_z_latent with shape (10000, 128)
Saved → ./h5_exports/train_img_z_latent.csv
Processing dataset: train_labels with shape (10000,)
Saved → ./h5_exports/train_labels.csv
Processing dataset: train_patterns with shape (10000,)
Saved → ./h5_exports/train_patterns.csv
Processing dataset: train_samples with shape (10000,)
Saved → ./h5_exports/train_samples.csv
Processing dataset: train_slides with shape (10000,)
Saved → ./h5_exports/train_slides.csv
Processing dataset: train_tiles with shape (10000,)
Saved → ./h5_exports/train_tiles.csv
Done! All datasets extracted.


hpc_dictionary malignant detail and non-malignant details

In [17]:




dfs = []
for f in ANNOTATION_FILES:
    df = pd.read_csv(f)
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
    df.columns = (df.columns.str.strip()
                             .str.replace(" ", "_")
                             .str.replace("-", "_")
                             .str.lower())
     #If there are duplicate column names, suffix them
    dups = df.columns[df.columns.duplicated()].tolist()
    if dups:
        new_cols = []
        counts = {}
        for c in df.columns:
            if c in counts:
                counts[c] += 1
                new_cols.append(f"{c}__dup{counts[c]}")
            else:
                counts[c] = 0
                new_cols.append(c)
        df.columns = new_cols  

    dfs.append(df)

#
merged = pd.concat(dfs, ignore_index=True)


print("Inflammation cols:", [c for c in merged.columns if "inflamm" in c])
print("Necrosis cols:", [c for c in merged.columns if "necro" in c])
print("Duplicated cols:", merged.columns[merged.columns.duplicated()].tolist())

merged = merged.rename(columns={
    "inflammation*": "inflammation_malignant",
    "inflammation*.1": "inflammation_non_malignant",
    "necrosis**": "necrosis_malignant",
    "necrosis**.1": "necrosis_non_malignant",
})


def majority_vote(values):
    vals = pd.Series(values).dropna().astype(str).str.strip()
    return vals.mode().iloc[0] if not vals.mode().empty else None

# majority-vote on *labels* (no mean)
consensus = merged.groupby("cluster").agg({
    "malignant_epithelium": majority_vote,
    "predominant_pattern": majority_vote,
    "second_pattern_(if_present)": majority_vote,
    "inflammation_malignant": majority_vote,
    "inflammation_non_malignant": majority_vote,
    "necrosis_malignant": majority_vote,
    "necrosis_non_malignant": majority_vote,
    "stroma_epithelium_ratio": majority_vote,
    "stromal_cellularity***": majority_vote,
    "other_comments": majority_vote,
    "tiles_contain_mostly": majority_vote,
    "second_most_common_feature": majority_vote,
    "other_notable_features_or_comments?": majority_vote,
    "percentage_cluster_purity": majority_vote,   # <-- label mode, not mean
}).reset_index()


consensus["malignant"] = consensus["malignant_epithelium"].str.lower().eq("yes")

consensus["inflammation"] = consensus.apply(
    lambda r:
        r["inflammation_malignant"]
        if r["malignant"]
        else r["inflammation_non_malignant"],
    axis=1
)

consensus["necrosis"] = consensus.apply(
    lambda r:
        r["necrosis_malignant"]
        if r["malignant"]
        else r["necrosis_non_malignant"],
    axis=1
)

consensus = consensus.drop(columns=[
    "inflammation_malignant",
    "inflammation_non_malignant",
    "necrosis_malignant",
    "necrosis_non_malignant",
])
# rename + boolean
consensus = consensus.rename(columns={
    "cluster": "hpc_id",
    "percentage_cluster_purity": "cluster_purity_label"
})
consensus["malignant"] = consensus["malignant_epithelium"].str.lower().eq("yes")
consensus = consensus.drop(columns=["malignant_epithelium"])

# Fix hpc_id (remove .0, convert to clean string)
consensus["hpc_id"] = ( consensus["hpc_id"] .apply(lambda x: str(int(float(x))) if pd.notna(x) else None))

# (optional) also provide a numeric for convenience derived from the voted label
def purity_label_to_num(s):
    if not isinstance(s, str):
        return None
    s = s.lower()
    if "high" in s: return 0.85
    if "moderate" in s or "50" in s: return 0.65
    if "low" in s: return 0.35
    return None

consensus["cluster_homogeneity"] = consensus["cluster_purity_label"].apply(purity_label_to_num)

consensus.to_csv("hpc_dict_main.csv", index=False)
print("Saved hpc_dict_main.csv with label + numeric purity")
print(consensus[["hpc_id","cluster_purity_label","cluster_homogeneity"]].head())

Inflammation cols: ['inflammation*', 'inflammation*.1']
Necrosis cols: ['necrosis**', 'necrosis**.1']
Duplicated cols: []
Saved hpc_dict_main.csv with label + numeric purity
  hpc_id cluster_purity_label  cluster_homogeneity
0      0         High (> 75%)                 0.85
1      1         High (> 75%)                 0.85
2      2         High (> 75%)                 0.85
3      3         High (> 75%)                 0.85
4      4         High (> 75%)                 0.85


In [18]:
print(consensus.columns.tolist())

['hpc_id', 'predominant_pattern', 'second_pattern_(if_present)', 'stroma_epithelium_ratio', 'stromal_cellularity***', 'other_comments', 'tiles_contain_mostly', 'second_most_common_feature', 'other_notable_features_or_comments?', 'cluster_purity_label', 'malignant', 'inflammation', 'necrosis', 'cluster_homogeneity']


In [19]:
consensus.to_csv("hpc_dict_main.csv", index=False)

In [24]:
# --- Parent table ---
parent = consensus[["hpc_id", "malignant", "cluster_homogeneity", "inflammation", "necrosis"]]

parent.to_csv("hpc_dict.csv", index=False)

# --- Malignant details ---
malignant_df = consensus[consensus["malignant"] == True][[
    "hpc_id",
    "predominant_pattern",
    "second_pattern_(if_present)",
    "stroma_epithelium_ratio",
    "stromal_cellularity***",
    "other_comments"
]].rename(columns={
    "second_pattern_(if_present)": "second_pattern",
    "stromal_cellularity***": "stromal_cellularity"
})
malignant_df.to_csv("malignant.csv", index=False)

# --- Non-malignant details ---
non_malignant_df = consensus[consensus["malignant"] == False][[
    "hpc_id",
    "tiles_contain_mostly",
    "second_most_common_feature",
    "other_notable_features_or_comments?"
]].rename(columns={
    "other_notable_features_or_comments?": "other_notable_features"
})
non_malignant_df.to_csv("non_malignant.csv", index=False)

tile_registry

In [62]:

# Load your CSV


labels.rename(columns={'leiden_2.5': 'hpc_id'}, inplace=True)

# Clean column names
labels.columns = labels.columns.str.strip().str.lower()

# If any missing columns, fill them in
labels["image_index"] = range(len(labels))
labels["h5_source_path"] = "subset_all_10000_images.h5"  # or your HPC path

# Save cleaned file
labels.to_csv("tile_reg.csv", index=False)
print(" Saved tile_reg_ready.csv")

 Saved tile_reg_ready.csv


hpl_profile_proportion

In [37]:

# # Check columns
# print(labels.columns.tolist())
# # Expected: ['samples', 'slides', 'tiles', 'leiden_2.5']

# # Rename for clarity
# df = label.rename(columns={"leiden_2.5": "hpc_id"})

# --- STEP 1: Count how many tiles per (sample, slide, hpc_id)
proportions = (
    labels.groupby(["samples", "slides", "hpc_id"], as_index=False)
    .size()
)

# --- STEP 2: Convert counts to proportions per (sample, slide)
proportions["proportion"] = proportions.groupby(
    ["samples", "slides"]
)["size"].transform(lambda x: x / x.sum())

# --- STEP 3: Clean up
proportions = proportions.drop(columns=["size"])

# --- STEP 4: Save to CSV
proportions.to_csv("hpl_profile_prop.csv", index=False)

print("Saved hpl_profile_prop.csv")
print(proportions.head())

Saved hpl_profile_prop.csv
        samples                   slides  hpc_id  proportion
0  TCGA-05-4245  TCGA-05-4245-01Z-00-DX1       2    1.000000
1  TCGA-05-4250  TCGA-05-4250-01Z-00-DX1       2    0.006211
2  TCGA-05-4250  TCGA-05-4250-01Z-00-DX1      10    0.012422
3  TCGA-05-4250  TCGA-05-4250-01Z-00-DX1      16    0.012422
4  TCGA-05-4250  TCGA-05-4250-01Z-00-DX1      21    0.006211


Hpl_profile summary

In [47]:


# Load the HPC label data


# Clean column names for consistency
labels.columns = labels.columns.str.strip().str.lower().str.replace(" ", "_")

# Now check which columns exist
print(labels.columns)

# Assuming columns include: 'samples', 'slides', 'tiles', 'leiden_2.5'
# If names differ, adjust below accordingly
group_cols = ["samples", "slides"]

# Compute total tiles per sample/slide
summary = labels.groupby(group_cols).agg(
    total_tiles=("tiles", "count"),
    dominant_hpc=("hpc_id", lambda x: x.value_counts().idxmax())
).reset_index()

# Add id and cancer_type
summary["id"] = range(1, len(summary) + 1)
summary["cancer_type"] = "LUAD"  # replace with real data if available

# Reorder columns
summary = summary[["id", "samples", "slides", "cancer_type", "total_tiles", "dominant_hpc"]]

# Save
summary.to_csv("hpl_summary.csv", index=False)
print("hpl_summary.csv created successfully!")
print(summary.head())


Index(['samples', 'slides', 'tiles', 'hpc_id', 'image_index',
       'h5_source_path'],
      dtype='object')
hpl_summary.csv created successfully!
   id       samples                   slides cancer_type  total_tiles  \
0   1  TCGA-05-4245  TCGA-05-4245-01Z-00-DX1        LUAD            1   
1   2  TCGA-05-4250  TCGA-05-4250-01Z-00-DX1        LUAD          161   
2   3  TCGA-05-4382  TCGA-05-4382-01Z-00-DX1        LUAD          202   
3   4  TCGA-05-4396  TCGA-05-4396-01Z-00-DX1        LUAD          321   
4   5  TCGA-05-4398  TCGA-05-4398-01Z-00-DX1        LUAD          504   

   dominant_hpc  
0             2  
1            47  
2            48  
3            52  
4            47  


hpc_survival

In [25]:

# Clean column names
survival.columns = (
    survival.columns.str.strip()
                    .str.lower()
                    .str.replace(" ", "_")
                    .str.replace("(", "", regex=False)
                    .str.replace(")", "", regex=False)
                    .str.replace("%", "", regex=False)
)

# Fix the unwanted leading hyphen and rename properly
survival = survival.rename(columns={"-log2p": "log2_p"})

# Rename covariate to hpc_id
survival = survival.rename(columns={"covariate": "hpc_id"})

# --- CRITICAL FIX ---
# Convert "HPC 56" → 56
survival["hpc_id"] = (
    survival["hpc_id"]
        .astype(str)
        .str.replace("hpc", "", case=False)
        .str.replace("[^0-9]", "", regex=True)
        .replace("", pd.NA)
        .astype("Int64")
)

# Optional sanity check
print("Unique HPC IDs:", survival["hpc_id"].dropna().unique()[:10])

# Save cleaned file
survival.to_csv("survival.csv", index=False)
print("Saved cleaned survival.csv")

Unique HPC IDs: <IntegerArray>
[56, 60, 0, 59, 69, 48, 16, 11, 25, 34]
Length: 10, dtype: Int64
Saved cleaned survival.csv


h_latent_for_kb

In [87]:
import pandas as pd

# Load latent file
latent = pd.read_csv("/Users/vaidehipandya/Desktop/Work/h5_exports/train_img_h_latent.csv")

# Rename latent columns
latent.columns = [f"latent_{i}" for i in range(latent.shape[1])]

# Merge side-by-side
merged = pd.concat([labels_ss.reset_index(drop=True), latent.reset_index(drop=True)], axis=1)
print("Merged successfully!")
print("Shape:", merged.shape)

# Clean whitespace
for col in ["samples", "slides", "tiles"]:
    if col in merged.columns:
        merged[col] = merged[col].astype(str).str.strip()

# Rename leiden_2.5 → hpc_id BEFORE auto detection
if "leiden_2.5" in merged.columns:
    merged = merged.rename(columns={"leiden_2.5": "hpc_id"})

# Auto-detect hpc column
hpc_col = None
for c in merged.columns:
    if "hpc" in c.lower():      # now picks up hpc_id directly
        hpc_col = c
        break

if hpc_col is None:
    raise KeyError("Could not find a column with 'hpc' in its name.")

# Final hpc_id column
merged["hpc_id"] = merged[hpc_col].astype(str).str.strip()

# Rename columns for DB consistency
merged = merged.rename(columns={
    "samples": "sample_id",
    "slides": "slide_id",
    "tiles": "tile_name"
})

# Add image_index column
merged["image_index"] = range(len(merged))

# Add h5 source path column
merged["h5_source_path"] = "subset_all_10000.h5"

# Save final file
merged.to_csv("h_latent.csv", index=False)
print("Saved: h_latent.csv")
print("Final shape:", merged.shape)
print("Columns:", merged.columns[:15].tolist(), "...")

Merged successfully!
Shape: (10000, 1028)
Saved: h_latent.csv
Final shape: (10000, 1030)
Columns: ['sample_id', 'slide_id', 'tile_name', 'hpc_id', 'latent_0', 'latent_1', 'latent_2', 'latent_3', 'latent_4', 'latent_5', 'latent_6', 'latent_7', 'latent_8', 'latent_9', 'latent_10'] ...


In [93]:


# Identify latent columns automatically
latent_cols = [c for c in merged.columns if c.startswith("latent_")]

# Convert to pgvector format: {x,y,z,...}
merged["embedding"] = merged[latent_cols].apply(
    lambda row: "{" + ",".join(map(str, row.values)) + "}", axis=1
)
# Replace curly braces with square brackets for pgvector
merged["embedding"] = (
    merged["embedding"]
    .astype(str)
    .str.replace("{", "[", regex=False)
    .str.replace("}", "]", regex=False)
)
df_final = merged[["sample_id", "slide_id", "tile_name", "hpc_id", "embedding"]]

# Save the cleaned file
df_final.to_csv("h_latent_pgsql.csv", index=False)
print(" File ready for pgvector import: h_latent_pgsql.csv")



 File ready for pgvector import: h_latent_pgsql.csv


In [95]:


old_dir = "/Users/vaidehipandya/Desktop/Work/old.csv"
new_dir = "/Users/vaidehipandya/Desktop/Work/new.csv"

old_files = sorted([f for f in os.listdir(old_dir) if f.endswith(".csv")])
new_files = sorted([f for f in os.listdir(new_dir) if f.endswith(".csv")])

print("Old CSVs:", old_files)
print("New CSVs:", new_files)

# Make sure filenames match
if old_files != new_files:
    print("⚠️ Filenames do NOT match!")
    print("Only in OLD:", set(old_files) - set(new_files))
    print("Only in NEW:", set(new_files) - set(old_files))
else:
    print("\n✔ Filenames match exactly.\n")

for fname in old_files:

    print(f"\n🔍 Comparing {fname} ...")

    old_path = os.path.join(old_dir, fname)
    new_path = os.path.join(new_dir, fname)

    old = pd.read_csv(old_path)
    new = pd.read_csv(new_path)

    # Sort columns and rows to remove ordering differences
    old_sorted = old.reindex(sorted(old.columns), axis=1)\
                    .sort_values(by=sorted(old.columns))\
                    .reset_index(drop=True)

    new_sorted = new.reindex(sorted(new.columns), axis=1)\
                    .sort_values(by=sorted(new.columns))\
                    .reset_index(drop=True)

    # If exactly equal
    if old_sorted.equals(new_sorted):
        print(f"   ✔ MATCH — {fname} is identical")
        continue

    # If shapes differ, print summary
    if old_sorted.shape != new_sorted.shape:
        print("   ✘ DIFFERENT SHAPE")
        print("     OLD:", old_sorted.shape)
        print("     NEW:", new_sorted.shape)

    # Identify column mismatches
    old_cols = set(old_sorted.columns)
    new_cols = set(new_sorted.columns)

    if old_cols != new_cols:
        print("   ⚠️ Column mismatch detected")
        print("     Only in OLD:", old_cols - new_cols)
        print("     Only in NEW:", new_cols - old_cols)

    # Try detailed compare if structures match
    if old_cols == new_cols:
        try:
            diff = old_sorted.compare(new_sorted, keep_shape=False, keep_equal=False)
            diff_file = f"diff_{fname}"
            diff.to_csv(diff_file, index=False)
            print(f"   ✘ DIFFERENT — Saved detailed diff → {diff_file}")
        except Exception as e:
            print("   ⚠️ Could not compare due to:", str(e))

Old CSVs: ['h_latent.csv', 'h_latent_pgsql.csv', 'hpc_dict.csv', 'hpc_dict_main.csv', 'hpl_profile_prop.csv', 'hpl_summary.csv', 'malignant.csv', 'non_malignant.csv', 'survival.csv', 'tile_reg.csv']
New CSVs: ['h_latent.csv', 'h_latent_pgsql.csv', 'hpc_dict.csv', 'hpc_dict_main.csv', 'hpl_profile_prop.csv', 'hpl_summary.csv', 'malignant.csv', 'non_malignant.csv', 'survival.csv', 'tile_reg.csv']

✔ Filenames match exactly.


🔍 Comparing h_latent.csv ...
   ✘ DIFFERENT SHAPE
     OLD: (142493, 1030)
     NEW: (10000, 1030)
   ⚠️ Could not compare due to: Can only compare identically-labeled (both index and columns) DataFrame objects

🔍 Comparing h_latent_pgsql.csv ...
   ✘ DIFFERENT — Saved detailed diff → diff_h_latent_pgsql.csv

🔍 Comparing hpc_dict.csv ...
   ✔ MATCH — hpc_dict.csv is identical

🔍 Comparing hpc_dict_main.csv ...
   ✔ MATCH — hpc_dict_main.csv is identical

🔍 Comparing hpl_profile_prop.csv ...
   ✔ MATCH — hpl_profile_prop.csv is identical

🔍 Comparing hpl_summary.csv 